In [2]:
import pandas as pd
import altair as alt

In [34]:
from ecostyles import EcoStyles
styles = EcoStyles()
styles.register_and_enable_theme(theme_name="article")

Source: https://www.ons.gov.uk/employmentandlabourmarket/peopleinwork/labourproductivity/datasets/subregionalproductivitylabourproductivityindicesbycombinedauthoritiesandeconomicenterpriseregions

In [14]:
a3 = pd.read_excel('labourproductivitycaeer3.xlsx', sheet_name='A3', skiprows=4, nrows=34)
a5 = pd.read_excel('labourproductivitycaeer3.xlsx', sheet_name='A5', skiprows=4, nrows=34)

# Melt both to long format
a3_melted = a3.melt(id_vars=['area_name', 'area_code'], var_name='year', value_name='cp_gva')
a5_melted = a5.melt(id_vars=['area_name', 'area_code'], var_name='year', value_name='cvm_index')

# Clean year columns
for df in [a3_melted, a5_melted]:
    df['year'] = pd.to_numeric(df['year'].astype(str).str.extract(r'(\d{4})')[0])

# Merge
merged = a3_melted.merge(a5_melted, on=['area_name', 'area_code', 'year'])

# Get 2022 anchor value from A3 for each region
anchor = merged[merged['year'] == 2022][['area_name', 'cp_gva']].rename(columns={'cp_gva': 'anchor_2022'})
merged = merged.merge(anchor, on='area_name')

# Get 2022 CVM index value for each region
cvm_base = merged[merged['year'] == 2022][['area_name', 'cvm_index']].rename(columns={'cvm_index': 'cvm_2022'})
merged = merged.merge(cvm_base, on='area_name')

# Reconstruct real £2022 series
merged['gva_real'] = merged['anchor_2022'] * (merged['cvm_index'] / merged['cvm_2022'])

In [27]:
hours

,area_code,area_name,Hours_2004,Hours_2005,Hours_2006,Hours_2007,Hours_2008,Hours_2009,Hours_2010,Hours_2011,...,Hours_2014,Hours_2015,Hours_2016,Hours_2017,Hours_2018,Hours_2019,Hours_2020,Hours_2021,Hours_2022,Hours_2023
0,UKX,United Kingdom less Extra-Regio,907931111,927870600,933957804,942790207,938434253,920281841,916228621,928360499,...,990512607,997197982,1021052944,1030145262,1039012521,1055242213,932135358,1010135088,1048589941,1055637266
1,E47000001,Greater Manchester,38227374,39132696,38899873,38547864,38266223,37828788,37952358,37696811,...,40714497,40334789,41799966,42822158,44238273,45556173,39538012,44796748,45264303,44642078
2,E47000002,South Yorkshire,17369526,17726016,17535204,17504845,17023138,16700058,16518706,17132154,...,17845202,17877133,18008830,18094374,18522152,18453509,17085685,18361454,19057762,18665043
3,E47000003,West Yorkshire,32372648,32705018,32782750,32614040,32185131,32088648,31858264,31807388,...,33275286,33866433,34830230,34786774,35708933,36024928,31755701,35062803,35699422,35320268
4,E47000004,Liverpool City Region,18743694,19133848,19316979,18887018,19286337,18528305,18525841,18505277,...,19738857,19639057,19891965,20541766,20138376,21829747,18562600,20096206,21330452,20827480
5,E47000006,Tees Valley,8504133,8362750,8558112,8553001,8601705,8349767,8413647,8226289,...,8378844,8400164,8291586,8371894,8443308,8519214,7669077,8665621,8670269,8564460
6,E47000007,West Midlands,38475229,39536082,39495626,40199348,38240824,36708258,36936374,37140053,...,39523928,41090515,40982509,40909394,41793414,42314814,37160089,39560876,41713162,43165430
7,E47000008,Cambridgeshire and Peterborough,11615451,12418130,13315428,13218429,13087977,13004521,12965527,12574331,...,13688650,14195136,14494921,14889477,15088960,15466748,13753355,14451613,15051994,15492751
8,E47000009,West of England,14866324,15274745,15369014,15372760,14821723,14943268,14905617,15278536,...,16290850,15926891,16208802,16765172,16968216,17365665,16317604,17489521,18542174,18462264
9,E47000012,York and North Yorkshire,11112496,11797693,11948981,12118443,11879470,12284352,12289648,12185420,...,13005449,13111976,12958936,12836316,13666662,12877915,11001356,12197532,12948372,14096037


In [47]:
# Filter to London components
london = merged[merged['area_name'].isin(['Inner London', 'Outer London'])].copy()

# Get hours worked from the workbook
hours = pd.read_excel('labourproductivitycaeer3.xlsx', 
                      sheet_name='Productivity Hours', skiprows=4, nrows=34)


hours_melted = hours.melt(id_vars=['area_name', 'area_code'], var_name='year', value_name='hours')
hours_melted['year'] = pd.to_numeric(hours_melted['year'].astype(str).str.extract(r'(\d{4})')[0])

# Merge hours onto london
london = london.merge(hours_melted[['area_name', 'year', 'hours']], 
                      left_on=['area_name', 'year'], right_on=['area_name', 'year'])

# Weighted average by year
london_agg = (
    london.groupby('year')
    .apply(lambda x: (x['gva_real'] * x['hours']).sum() / x['hours'].sum())
    .reset_index(name='gva_real')
)
london_agg['city_region_name'] = 'London'

/var/folders/qz/pj0lh7817m3c9ydwgfrtbtf00000gn/T/ipykernel_84157/2703129098.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: (x['gva_real'] * x['hours']).sum() / x['hours'].sum())


In [48]:
# The 6 non-London regions
keep = [
    'Greater Manchester',
    'South Yorkshire',
    'West Yorkshire',
    'Liverpool City Region',
    'West Midlands',
    'North East',
]

name_map = {
    'Greater Manchester': 'Greater Manchester',
    'South Yorkshire': 'South Yorkshire',
    'West Yorkshire': 'West Yorkshire',
    'Liverpool City Region': 'Liverpool',
    'West Midlands': 'West Midlands',
    'North East': 'North East',
}

other_regions = merged[merged['area_name'].isin(keep)].copy()
other_regions['city_region_name'] = other_regions['area_name'].map(name_map)
other_regions = other_regions[['year', 'city_region_name', 'gva_real']]

# London aggregated
london_agg['year'] = london_agg['year']  # already numeric, convert below

# Combine
chart_df = pd.concat([other_regions, london_agg[['year', 'city_region_name', 'gva_real']]])
chart_df['year'] = pd.to_datetime(chart_df['year'], format='%Y')

In [52]:
labelled = chart_df[chart_df['city_region_name'].isin(['London', 'Greater Manchester'])]
grey = chart_df[~chart_df['city_region_name'].isin(['London', 'Greater Manchester'])]
label_data = chart_df[chart_df['year'] == chart_df['year'].max()]
label_data = label_data[label_data['city_region_name'].isin(['London', 'Greater Manchester'])]

grey_lines = alt.Chart(grey).mark_line(color='#bbbbbb', strokeWidth=1.5).encode(
    x=alt.X('year:T', axis=alt.Axis(format='%Y', title=None, labelFontSize=14)),
    y=alt.Y('gva_real:Q',
            scale=alt.Scale(domain=[25, 55]),
            axis=alt.Axis(labelExpr="'£' + datum.value", labelFontSize=14, 
                          title='GVA per hour worked', titleFontSize=14)),
    detail='city_region_name:N',
)

colour_lines = alt.Chart(labelled).mark_line(strokeWidth=2.5).encode(
    x=alt.X('year:T'),
    y=alt.Y('gva_real:Q'),
    color=alt.Color('city_region_name:N', scale=alt.Scale(
        domain=['London', 'Greater Manchester'],
        range=['#179fdb', '#e6224b']
    ), legend=None),
)

labels = alt.Chart(label_data).mark_text(
    align='left', dx=6, fontSize=13, fontWeight='bold'
).encode(
    x=alt.X('year:T'),
    y=alt.Y('gva_real:Q'),
    text='city_region_name:N',
    color=alt.Color('city_region_name:N', scale=alt.Scale(
        domain=['London', 'Greater Manchester'],
        range=['#179fdb', '#e6224b']
    ), legend=None),
)

chart = (grey_lines + colour_lines + labels).properties(
    width=450,
    height=300,
).configure_view(
    strokeWidth=0
).configure_axis(
    grid=True, gridDash=[2, 2], gridColor='#dddddd'
)

chart

alt.LayerChart(...)

In [63]:
label_offsets = {
    'London': {'dx': -45, 'dy': -13},
    'Greater Manchester': {'dx': -120, 'dy': -10},
}

text_layers = [
    alt.Chart(label_data[label_data['city_region_name'] == name]).mark_text(
        align='left',
        fontSize=13,
        fontWeight='bold',
        dx=offsets['dx'],
        dy=offsets['dy'],
    ).encode(
        x=alt.X('year:T'),
        y=alt.Y('gva_real:Q'),
        text='city_region_name:N',
        color=alt.Color('city_region_name:N', scale=alt.Scale(
            domain=['London', 'Greater Manchester'],
            range=['#179fdb', '#e6224b']
        ), legend=None),
    )
    for name, offsets in label_offsets.items()
]

chart = alt.layer(grey_lines, colour_lines, *text_layers).properties(
    width=500,
    height=300,
).configure_view(
    strokeWidth=0
).configure_axis(
    grid=True, gridDash=[2, 2], gridColor='#dddddd'
)

chart

alt.LayerChart(...)

In [64]:
# Save to png
chart.save('Manchester_elections_fig1.png', scale_factor=2)
# Save to json
chart.save('Manchester_elections_fig1.json', scale_factor=2)